In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
np.random.seed(42)

n_samples = 20000

# Features (realistic ranges)

age = np.random.randint(21, 65, n_samples)

income = np.random.normal(60000, 15000, n_samples)     # annual income
credit_score = np.random.normal(650, 70, n_samples)    # typical credit score range
loan_amount = np.random.normal(20000, 8000, n_samples)
years_employed = np.random.randint(0, 30, n_samples)
debt_to_income = np.random.uniform(0.05, 0.6, n_samples)
savings_balance = np.random.normal(15000, 10000, n_samples)

# Linear decision function (true boundary)
z = (
    0.003 * income +
    0.01 * credit_score +
    0.5 * years_employed +
    0.0005 * savings_balance
    - 0.004 * loan_amount
    - 15 * debt_to_income
    - 8
)

# Labels
y = (z > 0).astype(int)

# Create DataFrame
dataset = pd.DataFrame({
    "age": age,
    "income": income,
    "credit_score": credit_score,
    "loan_amount": loan_amount,
    "years_employed": years_employed,
    "debt_to_income": debt_to_income,
    "savings_balance": savings_balance,
    "loan_approved": y
})

print(dataset.head())
print("\nDataset shape:", dataset.shape)

   age        income  credit_score   loan_amount  years_employed  \
0   59  87943.036821    652.122671  32532.317727              11   
1   49  61869.950901    672.209246  17863.881039               2   
2   35  76491.956571    736.933527  11290.461215              20   
3   63  54725.097242    705.059945  13443.360531               4   
4   28  77775.104754    602.329323  35582.358880               4   

   debt_to_income  savings_balance  loan_approved  
0        0.246817      5396.091534              1  
1        0.249496     17898.093571              1  
2        0.341243     29603.774167              1  
3        0.344308      7933.857497              1  
4        0.228200     22199.160439              1  

Dataset shape: (20000, 8)


In [14]:
def support_vector_machines_classifier(dataset,unseen_dataset):
    dataset = dataset.sample(frac=1, random_state=42).reset_index(drop=True)

    split=int(0.7*len(dataset))

    train_data=dataset.iloc[:split]
    test_data=dataset.iloc[split:]

    x_train=train_data.iloc[:,:-1]
    y_train=train_data.iloc[:,-1].values

    x_test=test_data.iloc[:,:-1]
    y_test=test_data.iloc[:,-1].values

    mean=x_train.mean()
    std=x_train.std()

    x_train=(x_train-mean)/std
    x_test=(x_test-mean)/std

    x_train=x_train.values
    x_test=x_test.values

    y_train = np.where(y_train==0,-1,1)
    y_test = np.where(y_test==0,-1,1)

    n_weights=len(dataset.columns)-1
    weights = np.random.randn(n_weights) * 0.01
    bias=np.random.rand()

    learning_rate=0.01
    lamda=0.01
    epochs=500

    for epoch in range(1,epochs+1):
        indices = np.random.permutation(len(x_train))
        x_shuffled = x_train[indices]
        y_shuffled = y_train[indices]

        for x,y in zip(x_shuffled,y_shuffled):
            z = (np.dot(weights,x) + bias) * y

            if z>=1:
                weights = weights - learning_rate * 2 * lamda * weights

            else:
                weights = weights - learning_rate * (-y * x + 2 * lamda * weights)
                bias = bias + learning_rate * y
        
        if epoch % 50 == 0:
            z = y_train * (np.dot(x_train, weights) + bias)
            loss = np.mean(np.maximum(0, 1 - z))
            print(f'Epoch: {epoch}, Loss: {loss}')
        
    predictions = np.sign(np.dot(x_test,weights) + bias)

    accuracy = np.sum(predictions == y_test) / len(y_test)
    print(f'\nAccuracy: {accuracy}', end='\n\n')

    unseen_dataset = (unseen_dataset - mean) / std
    unseen_dataset = unseen_dataset.values

    unseen_predictions = np.sign(np.dot(unseen_dataset,weights) + bias)

    return unseen_predictions


In [21]:
# number of samples per class
n = 10

# Positive leaning samples (likely +1)
positive_samples = pd.DataFrame({
    "age": np.random.randint(30, 60, n),
    "income": np.random.uniform(70000, 120000, n),
    "credit_score": np.random.uniform(700, 800, n),
    "loan_amount": np.random.uniform(5000, 15000, n),
    "years_employed": np.random.randint(10, 30, n),
    "debt_to_income": np.random.uniform(0.05, 0.25, n),
    "savings_balance": np.random.uniform(20000, 60000, n)
})

# Negative leaning samples (likely -1)
negative_samples = pd.DataFrame({
    "age": np.random.randint(21, 40, n),
    "income": np.random.uniform(20000, 40000, n),
    "credit_score": np.random.uniform(500, 620, n),
    "loan_amount": np.random.uniform(25000, 50000, n),
    "years_employed": np.random.randint(0, 5, n),
    "debt_to_income": np.random.uniform(0.5, 0.8, n),
    "savings_balance": np.random.uniform(0, 5000, n)
})

# combine
unseen_dataset = pd.concat([positive_samples, negative_samples]).reset_index(drop=True)
unseen_dataset = unseen_dataset.sample(frac=1, random_state=42).reset_index(drop=True)

print(unseen_dataset)

    age         income  credit_score   loan_amount  years_employed  \
0    53   97110.512268    714.533230  12760.471307              23   
1    33   22801.225005    507.877646  27481.353045               0   
2    23   37591.092372    608.665666  32132.829006               0   
3    37  105581.886887    746.514907   9119.408385              19   
4    42   78380.123752    731.395248   7019.248433              21   
5    37   78704.910330    792.925969   9705.511269              21   
6    37   38585.599601    584.071459  41326.340821               1   
7    54   93703.848384    751.412796  12827.940979              23   
8    30   29042.830278    518.670283  38089.308506               4   
9    21   26958.573228    525.299693  38439.158645               1   
10   27   29340.072433    607.528099  42728.264884               2   
11   55  117222.092044    778.875273  12521.907461              13   
12   44   70051.239075    736.342283   7055.140566              24   
13   24   29992.9151

In [22]:
predictions = support_vector_machines_classifier(dataset, unseen_dataset)
print('Predictions:\n',predictions)

Epoch: 50, Loss: 0.04364156241135428
Epoch: 100, Loss: 0.04268318228116144
Epoch: 150, Loss: 0.040759110882449476
Epoch: 200, Loss: 0.04206737641242322
Epoch: 250, Loss: 0.040775084096311955
Epoch: 300, Loss: 0.04075482915581053
Epoch: 350, Loss: 0.04062882428008971
Epoch: 400, Loss: 0.04124440058710562
Epoch: 450, Loss: 0.04064544768347699
Epoch: 500, Loss: 0.04037627187992577

Accuracy: 0.9791666666666666

Predictions:
 [ 1. -1.  1.  1.  1.  1. -1.  1. -1. -1. -1.  1.  1.  1.  1. -1.  1.  1.
 -1.  1.]
